In [ ]:
import pandas as pd
import os
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

# Clé API OpenAI
os.environ["OPENAI_API_KEY"] = "sk-ta-cle-api-ici"

print("Librairies RAG chargées")

In [ ]:
# 1. Charger tes métriques de forme actuelles
daily = pd.read_csv("../data/processed/daily_features.csv")
current_state = daily.iloc[-1] # Le dernier jour enregistré

# 2. Charger tes dernières anomalies (fatigue détectée)
anomalies = pd.read_csv("../data/processed/ml_results.csv")
recent_anomalies = anomalies[anomalies["anomaly"] == -1].tail(3)

# 3. Créer le "Dossier Médical/Sportif" de l'athlète à donner à l'IA
athlete_context = f"""
ÉTAT DE FORME ACTUEL DE HICHAM :
- CTL (Forme de fond) : {current_state['CTL']} (Plus c'est haut, mieux c'est)
- ATL (Fatigue immédiate) : {current_state['ATL']}
- TSB (Fraîcheur / Forme du jour) : {current_state['TSB']} (Si négatif = fatigué, si positif = reposé)
- Charge de la dernière séance (TRIMP) : {current_state['TRIMP']}

DERNIÈRES ANOMALIES DÉTECTÉES PAR LE ML :
{recent_anomalies[['date', 'distance_km', 'fc_moyenne', 'vitesse_moy']].to_string(index=False)}
"""

print("Contexte Data généré pour le LLM :")
print(athlete_context)

In [ ]:
# Base de connaissances théoriques sur l'entraînement
training_docs = [
    "La règle des 80/20 stipule que 80% de l'entraînement doit se faire en endurance fondamentale (basse fréquence cardiaque) et 20% à haute intensité.",
    "Un TSB (Training Stress Balance) inférieur à -20 indique un risque très élevé de blessure ou de surentraînement. Il faut privilégier la récupération active ou le repos absolu.",
    "Un TSB compris entre +5 et +25 est la zone de 'tapering' (affûtage) idéale juste avant une compétition pour optimiser la performance.",
    "Si l'algorithme d'anomalie détecte une fréquence cardiaque anormalement élevée pour une vitesse faible, c'est un signe précurseur de maladie, de déshydratation ou de manque de sommeil profond.",
    "Une séance de récupération active ne doit pas dépasser 45 minutes avec une fréquence cardiaque moyenne inférieure à 130 bpm."
]

# Création de la base vectorielle locale avec ChromaDB
vectorstore = Chroma.from_texts(
    texts=training_docs,
    embedding=OpenAIEmbeddings(),
    collection_name="athlete_knowledge_base"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("Base de connaissances vectorisée (ChromaDB) prête.")

In [ ]:
# Le "Cerveau" de ton agent
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.2)

# Le Prompt System ultra-ciblé
template = """Tu es 'AthleteAI', un coach d'athlétisme d'élite expert en Data Science sportive.
Tu parles directement à Hicham, ton athlète.

Voici son état de forme actuel issu de nos modèles de Machine Learning :
{athlete_context}

Voici les principes scientifiques pertinents pour sa question :
{context}

Question de Hicham : {question}

Consignes pour ta réponse :
1. Sois direct, professionnel, mais empathique.
2. Utilise toujours ses données réelles (CTL, TSB) pour justifier ta réponse.
3. Fais le lien entre la théorie scientifique et ses données de Machine Learning.
4. Propose une recommandation claire pour son entraînement de demain.

Réponse du Coach :"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# La chaîne RAG complète (LangChain Expression Language)
rag_chain = (
    {"context": retriever | format_docs, "athlete_context": lambda x: athlete_context, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("Agent LangChain instancié et prêt à l'emploi.")

In [ ]:
# Question 1 : Risque de blessure
question_1 = "Coach, j'ai envie de faire une sortie longue de 20km demain avec beaucoup de dénivelé. Qu'en penses-tu au vu de ma forme actuelle ?"
print(f"Hicham : {question_1}\n")
print(f"AthleteAI : {rag_chain.invoke(question_1)}")
print("-" * 80)

# Question 2 : Explication des anomalies ML
question_2 = "L'algorithme d'anomalie a flaggé mes séances récentes. Pourquoi ? Dois-je m'inquiéter ?"
print(f"Hicham : {question_2}\n")
print(f"AthleteAI : {rag_chain.invoke(question_2)}")